# 📓 04. Activity Contributions

### 🧮 Objective: Calculate activity scores for Combat, Collection, and Exploration.


In [1]:
# pip install pandas numpy

### 🧹 Output Artifacts:
- `../../data/processed/4_activity_contributions.csv`


In [2]:
import pandas as pd
import numpy as np

INPUT_PATH = '../../data/processed/3_normalized_telemetry.csv'
OUTPUT_PATH = '../../data/processed/4_activity_contributions.csv'

print("Libraries imported")

Libraries imported


In [3]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


## 🔥 CRITICAL BUG HISTORY: The 'Raw Sum' Disaster

> **The Bug:** Originally, the column selector used a loose filter that included both normalized data AND original `rawJson.*` columns.
>
> **The Math Error:**
> `score_explore = normalized_dist(0.5) + raw_dist(4500.0) = 4500.5`
> This massive raw value dwarfed Combat (0.5) and Collection (0.5).
>
> **The Consequence:** It appeared players ignored combat, when in reality the math was biased by measurement units (meters vs kills). We now strictly filter columns.



In [4]:
# =============================================================================
# ACTIVITY SCORE CALCULATION — v2.2 (Revised Formula)
# =============================================================================
#
# CHANGES FROM v2.1:
# 1. AVERAGES instead of sums (carried from v2.1): each category is divided
#    by its feature count. Equal ceiling of 1.0 for all archetypes.
#
# 2. timeOutOfCombat REMOVED from Exploration (carried from v2.1):
#    - It accumulated passively for any player not in combat — including
#      combat-intent players who simply had no enemies available to fight.
#    - It is the arithmetic complement of timeInCombat (they sum to session
#      duration), making them redundant and inversely correlated.
#    - Exploration is now measured by ACTIVE signals only.
#
# 3. v2.2 NEW DERIVED FEATURES (computed in notebook 03):
#    damage_per_hit added to Combat (5th feature) — captures sniper/heavy-weapon
#    archetype; raw enemiesHit alone underrepresents low-hit/high-damage players.
#    Formula: damageDone / max(enemiesHit, 1)
#
#    pickup_attempt_rate added to Collection (4th feature) — measures deliberate
#    collection intent; explorers passing items incidentally have low rate,
#    true collectors actively attempting pickups have high rate.
#    Formula: pickupAttempts / max(timeNearInteractables, 1)
#
# FEATURE GROUPS (all columns are normalized [0,1] from notebook 03):
#   Combat (5):     enemiesHit, damageDone, timeInCombat, kills, damage_per_hit
#   Collection (4): itemsCollected, pickupAttempts, timeNearInteractables, pickup_attempt_rate
#   Exploration (2): distanceTraveled, timeSprinting
# =============================================================================

combat_features  = ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'damage_per_hit']
collect_features = ['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'pickup_attempt_rate']
explore_features = ['distanceTraveled', 'timeSprinting']  # timeOutOfCombat excluded

# Verify all expected columns are present before computing
missing = [f for f in combat_features + collect_features + explore_features if f not in df.columns]
if missing:
    raise KeyError(f"Missing expected columns in normalized data: {missing}\n"
                   f"Ensure notebook 03 was run first and saved damage_per_hit + pickup_attempt_rate.")

df['score_combat']  = df[combat_features].mean(axis=1)   # avg of 5, range [0, 1]
df['score_collect'] = df[collect_features].mean(axis=1)  # avg of 4, range [0, 1]
df['score_explore'] = df[explore_features].mean(axis=1)  # avg of 2, range [0, 1]

df['score_total'] = df['score_combat'] + df['score_collect'] + df['score_explore']

print("Calculated activity scores (v2.2 — averages, derived features, timeOutOfCombat excluded)")
print(f"score_combat  range: [{df['score_combat'].min():.3f}, {df['score_combat'].max():.3f}]")
print(f"score_collect range: [{df['score_collect'].min():.3f}, {df['score_collect'].max():.3f}]")
print(f"score_explore range: [{df['score_explore'].min():.3f}, {df['score_explore'].max():.3f}]")

Calculated activity scores (v2.2 — averages, derived features, timeOutOfCombat excluded)
score_combat  range: [0.000, 0.892]
score_collect range: [0.000, 0.723]
score_explore range: [0.000, 1.000]


In [5]:
# Calculate percentages
df['pct_combat'] = df['score_combat'] / df['score_total'].replace(0, np.nan)
df['pct_collect'] = df['score_collect'] / df['score_total'].replace(0, np.nan)
df['pct_explore'] = df['score_explore'] / df['score_total'].replace(0, np.nan)

# Fill NaN (no activity) with equal distribution
df[['pct_combat', 'pct_collect', 'pct_explore']] = df[['pct_combat', 'pct_collect', 'pct_explore']].fillna(1/3)

print("Calculated activity percentages")

Calculated activity percentages


In [6]:
# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n Saved to {OUTPUT_PATH}")


 Saved to ../../data/processed/4_activity_contributions.csv
